# Data Exploration


## 1. Load and inspect Data
In this step, I will load the raw CSV file into a Pandas Dataframe. The goal is to inspect its structure and confirm the dataset is correctly balanced,
ensuring the classifier will be trained on well-balanced dataset, preventing model biases.


In [1]:
# Installation of the libraries requiered for the project
!pip install -q pandas numpy scikit-learn matplotlib SpaCy beautifulsoup4

In [2]:
## Import commands
import pandas as pd 
import spacy
from bs4 import BeautifulSoup

In [3]:
# Load the dataset
df = pd.read_csv('../data/IMDB Dataset.csv')

# Display the Data
print('---DATA HEAD(First 5 rows)---')
print(df.head())

print('\n--- SHAPE (rows,columns)---')
print(df.shape)

print('\n---SENTIMENT COUNTS (checking for balance)---')
print(df['sentiment'].value_counts())

---DATA HEAD(First 5 rows)---
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

--- SHAPE (rows,columns)---
(50000, 2)

---SENTIMENT COUNTS (checking for balance)---
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


## Initial Findings:
* The dataset contains 50,000 rows and 2 columns: 'review' and 'sentiment'.
* The dataset is perfectly balanced with 25,000 positive and 25,000 negative reviews. This is perfect for training an unbiased model.
* The dataset contains HTML tags and hyphenated words that must be removed in the cleaning phase.

## 2.Build the preprocessing pipeline
After the initial inspection of the data, now is time for one of the most important steps: Cleaning and standarize the data, for the vectorizer.
The Data exploration in step 1 revealed that the text contains raw HTML tags (Example on the row 1 of the displayed data).
Based on further analysis, a simple stop-word removal list would incorrectly remove important context words like intensifiers ("very") or negations ("not")
A more robust, professional approach is to **let the TF-IDF vectorizer handle this automatically**.

The Preprocessing pipeline will be simpler but powerfull:

1. **Remove all HTML tags** to clean the raw text.

2. **Lemmatize all text** each word to its root form to standardize the vocabulary.

3. **Do not remove stop words** remove common stop words could lead to confusing analysis, this project requires a high level solution for handling stop words.
By combining TF-IDF with N-grams (pairs and triplets of words), the vectorizer itself will learn that common phrases like "of the" are unimportant (low score) while critical phrases like "not good" are very important (high score).


In [4]:
print('Loading medium spaCy model(en_core_web_md)...')
# Load the medium model of Spacy 
nlp = spacy.load('en_core_web_md', disable=['parser','ner'])
print('Model loaded.')

def preprocess_review(review_text):
    """
    This function takes a raw review string and performs a 3-step cleaning process:
    1.Removes HTML tags using BeautifulSoup
    2.Tokenizes and lemmatizes the text.
    3.Returns a single string of cleaned, lemmatized tokens.
    """
    # 1.Remove HTML tags
    soup = BeautifulSoup(review_text,'html.parser')
    clean_text = soup.get_text()

    # 2. Process the cleaned text with SpaCy
    doc = nlp(clean_text.lower()) # Convert the text to lower case

    clean_tokens = []

    # 3. Loop and apply the custom rules

    for token in doc:
        
        # Check that isn't a punctuaction and it's not just whitespace
        if (not token.is_punct) and (not token.is_space):

            # Get the lemma, AND manually strip any ('-') from the edges.
            cleaned_lemma = token.lemma_.strip('-')
            # finally check: Only add the token if it's not an empty string
            if cleaned_lemma:
                clean_tokens.append(cleaned_lemma)

    # Return the cleaned tokens as a single string (Ready for the vectorizer)
    return " ".join(clean_tokens)

print("Preprocessing function created successfully.")
            
    

Loading medium spaCy model(en_core_web_md)...
Model loaded.
Preprocessing function created successfully.


## Test of the function
Now I will test the new created function, with one of the first reviews, in this case Row 1 ( Review that contains the HTML Tag)

In [5]:
# Get a sample review from the DataFrame
sample_review = df['review'][1]
print("Original Review")
print(sample_review)

# Run the function on the review
cleaned_sample = preprocess_review(sample_review)

print("\n Cleaned Review")
print(cleaned_sample)

Original Review
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terri

## Application of the function in the Dataset


In [ ]:
print("Applying preprocessing to all 50.000 review's ...")

# Create a new cleaned_review column by applying the function to every row in the 'review' column
df['cleaned_review'] = df['review'].apply(preprocess_review)
print('Preprocessing of the Dataset Complete!')


Applying preprocessing to all 50.000 review's ...


In [8]:
# Save the FULL cleaned data to a new CSV file in the data folder
df.to_csv('../data/reviews_cleaned_FULL.csv', index = False)

print("Cleaned Dataset saved successfully!")

Cleaned Dataset saved successfully!


## Step 1 & 2 complete.   
The data has been successfully loaded, explored and cleaned. A final preprocessed sample of 1,000 revies has been saved to 'reviews_cleaned_sample.csv' and is now ready for the modeling phase in the next notebook.